# LG Aimers — 투수 제구 Bayesian Shrinkage(C 방식) 실험

기존 Top20 HGB에서 raw `asof_pitcher_success_rate`와 Bayesian shrinkage `pitcher_success_shrunk`를 비교합니다.

- Train: 2019–2023
- Validation: 2024
- Primary metric: Brier Score
- Full run: HGB 350 iterations
- 비교: raw Top20 / C-shrunk Top20 / C-shrunk + reliability


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. GitHub 저장소 준비
Private repo이면 Colab Secrets에 `GITHUB_TOKEN` 또는 `GH_TOKEN`을 등록해 두세요. 토큰은 출력하지 않습니다.


In [ ]:
from pathlib import Path
import os, subprocess, base64

REPO = Path('/content/lg_aimers_experiment_lab')
BRANCH = 'agent/pitcher-shrinkage-c'
URL = 'https://github.com/tswaincae1221/lg_aimers_experiment_lab.git'

token = None
try:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'GH_TOKEN'):
        try:
            token = userdata.get(key)
            if token: break
        except Exception:
            pass
except Exception:
    pass

env = os.environ.copy()
if token:
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env['GIT_CONFIG_COUNT'] = '1'
    env['GIT_CONFIG_KEY_0'] = 'http.extraHeader'
    env['GIT_CONFIG_VALUE_0'] = f'AUTHORIZATION: basic {auth}'

if REPO.exists():
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH], check=True, env=env)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH], check=True, env=env)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH], check=True, env=env)
else:
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',URL,str(REPO)], check=True, env=env)

subprocess.run(['git','-C',str(REPO),'branch','--show-current'], check=True)


In [ ]:
%cd /content/lg_aimers_experiment_lab
!pip -q install pandas numpy scikit-learn joblib


## 2. 데이터 경로 확인
Drive에서 `train.csv`, `trackman_history.csv`를 자동 탐색합니다. 같은 이름의 파일이 여러 개면 직접 경로를 지정하세요.


In [ ]:
from pathlib import Path
MYDRIVE = Path('/content/drive/MyDrive')

def find_unique(name):
    hits = [p for p in MYDRIVE.rglob(name) if p.is_file()]
    print(name, len(hits), 'found')
    for p in hits[:20]: print(' -', p)
    return hits[0] if len(hits) == 1 else None

TRAIN = find_unique('train.csv')
TRACKMAN = find_unique('trackman_history.csv')
# 여러 개가 나오면 아래처럼 직접 지정하세요.
# TRAIN = Path('/content/drive/MyDrive/.../train.csv')
# TRACKMAN = Path('/content/drive/MyDrive/.../trackman_history.csv')
assert TRAIN is not None, 'TRAIN 경로를 직접 지정하세요.'
assert TRACKMAN is not None, 'TRACKMAN 경로를 직접 지정하세요.'
MAPPING = REPO / 'resources/pitcher_trackman_mapping.csv'
assert MAPPING.is_file()
print('TRAIN=', TRAIN)
print('TRACKMAN=', TRACKMAN)


## 3. 실행 설정
처음에는 `MODE='quick'`로 확인하고, 성공하면 `MODE='full'`로 바꿔 다시 실행하세요.


In [ ]:
MODE = 'quick'  # quick 또는 full
OUTPUT_ROOT = Path('/content/drive/MyDrive/aimers_data/results/pitcher_shrinkage_c')
OUTPUT_DIR = OUTPUT_ROOT / MODE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cache_candidates = []
for name in ('hgb_feature_selection_v2_full', 'hgb_feature_selection_v2'):
    cache_candidates += [p for p in MYDRIVE.rglob(name) if p.is_dir()]
CACHE_BASE = cache_candidates[0] if cache_candidates else None
print('OUTPUT_DIR=', OUTPUT_DIR)
print('CACHE_BASE=', CACHE_BASE)


## 4. 실험 실행
C 방식은 `p_shrunk = (n*p_raw + m*p_prior)/(n+m)`이며, prior는 예측 시즌보다 이전 시즌만 사용합니다.


In [ ]:
import subprocess, sys
cmd = [
    sys.executable, '-m', 'src.hgb_pitcher_shrinkage_experiment',
    '--train', str(TRAIN), '--trackman', str(TRACKMAN),
    '--mapping', str(MAPPING), '--output-dir', str(OUTPUT_DIR),
    '--validation-season', '2024'
]
if CACHE_BASE is not None:
    cmd += ['--trackman-cache-dir', str(CACHE_BASE)]
if MODE == 'quick':
    cmd += ['--strengths','10','50','100','--max-iter','120','--max-rows-per-season','5000','--max-trackman-rows','300000']
else:
    cmd += ['--strengths','5','10','20','30','50','75','100','150','200','--max-iter','350']
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(REPO))


## 5. 결과 확인
`delta_vs_raw < 0`이면 C 방식이 raw Top20보다 개선된 것입니다. 특히 `n<20`, `20<=n<50` 구간을 확인하세요.


In [ ]:
import pandas as pd
from IPython.display import display
scores = pd.read_csv(OUTPUT_DIR / 'pitcher_shrinkage_scores.csv')
raw_brier = float(scores.loc[scores['variant'].eq('raw_top20'), 'brier'].iloc[0])
scores['delta_vs_raw'] = scores['brier'] - raw_brier
display(scores.sort_values('brier'))

segments = pd.read_csv(OUTPUT_DIR / 'pitcher_shrinkage_segment_scores.csv')
display(segments.sort_values(['pitcher_n_bucket','brier']))
display(segments.pivot_table(index=['variant','shrinkage'], columns='pitcher_n_bucket', values='brier', aggfunc='first'))


결과는 Drive의 `aimers_data/results/pitcher_shrinkage_c/<MODE>`에 저장됩니다. Quick 성공 후 Full을 실행하세요.
